In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [38]:
train_part1 = pd.read_parquet("../data/train_part_3.parquet", engine='fastparquet')

In [39]:
train_part1.info()

<class 'pandas.DataFrame'>
RangeIndex: 28500849 entries, 0 to 28500848
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   customer_id                 int64  
 1   event_id                    int64  
 2   event_dttm                  object 
 3   event_type_nm               int32  
 4   event_desc                  int32  
 5   channel_indicator_type      int32  
 6   channel_indicator_sub_type  int32  
 7   operaton_amt                float64
 8   currency_iso_cd             float64
 9   mcc_code                    object 
 10  pos_cd                      float64
 11  accept_language             object 
 12  browser_language            object 
 13  timezone                    float64
 14  session_id                  float64
 15  operating_system_type       float64
 16  battery                     object 
 17  device_system_version       object 
 18  screen_size                 object 
 19  developer_tools             ob

In [40]:
def most_popular(dataframe):            
    cols = dataframe.columns
    for i in cols:
        print(f"{i}:")
        print(f"{dataframe[i].mode(dropna=False)}\n")

# most_popular(train_part1)

In [41]:
# cols = train_part1.columns
# for i in range(23):
#     print(f"{train_part1[cols[i]].value_counts(dropna=False)}\n")

In [42]:
# delete = ["accept_language", "browser_language", "timezone", "session_id", "operating_system_type", \
#         "battery", "device_system_version", "screen_size"]
delete = ["accept_language", "browser_language"]

train_part1.drop(columns=delete, inplace=True)

In [43]:
#########
columns = ["event_type_nm", "channel_indicator_type", "channel_indicator_sub_type", "currency_iso_cd", "mcc_code", "pos_cd", "timezone",\
           "developer_tools", "phone_voip_call_state", "web_rdp_connection", "compromised", "operating_system_type", "event_desc"]
for i in columns:
    train_part1[i] = train_part1[i].fillna(value=-1, inplace=False)
    train_part1[i] = train_part1[i].astype(dtype="int16")

In [44]:
# columns = ["event_type_nm", "channel_indicator_type", "channel_indicator_sub_type", "currency_iso_cd", "mcc_code", "pos_cd", \
#            "developer_tools", "phone_voip_call_state", "web_rdp_connection", "compromised"]

# for i in columns:
#     train_part1[i] =  train_part1[i].fillna(value=-99)
#     train_part1[i] = train_part1[i].astype(dtype="int16")
#     train_part1.loc[train_part1[i] == -99, i] = None

In [45]:
train_part1["Hour"] = pd.to_datetime(train_part1["event_dttm"]).dt.hour.astype(dtype="int16")
# train_part1.drop(columns="event_dttm", inplace=True)

#### Добавим таргет и преобразуем его

In [46]:
labels = pd.read_parquet("../data/train_labels.parquet", engine='fastparquet')

In [47]:
train_part1 = train_part1.merge(labels, how='left', on=['customer_id', 'event_id'])

In [48]:
train_part1["target"].value_counts(dropna=False)

target
NaN    28471979
1.0       16897
0.0       11973
Name: count, dtype: int64

In [49]:
###########
train_part1["target"] = train_part1["target"].fillna(0, inplace=True).astype(dtype="int16")

/tmp/ipykernel_26335/3661410496.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  train_part1["target"] = train_part1["target"].fillna(0, inplace=True).astype(dtype="int16")


In [50]:
train_part1["target"].value_counts()

target
0    28483952
1       16897
Name: count, dtype: int64

In [51]:
# train_part1.drop(columns=["customer_id", "event_id"], inplace=True)
train_part1.drop(columns=["event_id"], inplace=True)

In [52]:
train_part1.info()

<class 'pandas.DataFrame'>
RangeIndex: 28500849 entries, 0 to 28500848
Data columns (total 22 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   customer_id                 int64  
 1   event_dttm                  object 
 2   event_type_nm               int16  
 3   event_desc                  int16  
 4   channel_indicator_type      int16  
 5   channel_indicator_sub_type  int16  
 6   operaton_amt                float64
 7   currency_iso_cd             int16  
 8   mcc_code                    int16  
 9   pos_cd                      int16  
 10  timezone                    int16  
 11  session_id                  float64
 12  operating_system_type       int16  
 13  battery                     object 
 14  device_system_version       object 
 15  screen_size                 object 
 16  developer_tools             int16  
 17  phone_voip_call_state       int16  
 18  web_rdp_connection          int16  
 19  compromised                 in

Сравню некоторые ранее удаленные столбцы для классов 0 и 1

In [53]:
train_part1.loc[train_part1["target"] == 1].tail(5)

,customer_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,...,operating_system_type,battery,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,Hour,target
17702078,124265584425399,2025-04-26 11:25:02,7,56,4,15,NaN,-1,-1,-1,...,-1,None,13,1080x2208,0,0,-1,0,11,1
17702088,124265584425399,2025-04-26 11:54:39,7,56,4,15,NaN,-1,-1,-1,...,-1,None,13,1080x2208,0,0,-1,0,11,1
17702162,124265584425436,2024-10-19 14:24:12,14,42,3,4,1982000.0,0,-1,-1,...,9,20%,None,None,-1,-1,0,-1,14,1
17702565,124265584425461,2025-01-07 16:11:40,7,56,4,15,NaN,-1,-1,-1,...,-1,None,14,1080x2168,0,-1,-1,0,16,1
17702569,124265584425461,2025-01-08 15:52:31,7,56,4,15,NaN,-1,-1,-1,...,-1,None,14,1080x2168,0,-1,-1,0,15,1


Получили обработанный набор данных, теперь можно переходить к обучению моделей.

In [54]:
train_part1.to_parquet("../ClearData_for_part_4/train_part3.parquet", index=False, engine="fastparquet")